# Streaming patterns

Three reasons to stream:

1. **Latency UX.** First-token latency is what users feel; total time matters less.
2. **Backpressure.** You can stop a runaway generation early.
3. **Progressive parsing.** Render structured fields as they complete, instead of blocking on the full payload.

We cover three patterns: a synchronous token loop, an async generator, and partial-JSON parsing.

**Note.** This notebook works fully offline by replaying a cached non-streaming response and slicing it into chunks. With real API keys, the same code paths run against live streaming endpoints via `litellm.completion(..., stream=True)`.

In [1]:
# %% Notebook bootstrap — never touches API keys directly.
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# CI / offline mode: replay cached responses instead of calling out.
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


LLM_CACHE_ONLY = 1


In [2]:
from shared.llm import Message, complete
MODEL = 'openai/gpt-4o-mini'
NS = '00-foundations/03-streaming-patterns'

full = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Explain streaming briefly.'),
        Message(role='user', content='Why stream LLM output?'),
    ],
).content
print('full length:', len(full), 'chars')

full length: 320 chars


## 1) Synchronous token loop
We simulate streaming by chunking the cached response. In live mode you'd iterate `for chunk in litellm.completion(..., stream=True): print(chunk.choices[0].delta.content)`.

In [3]:
import time, sys
def fake_stream(text: str, chunk_chars: int = 16):
    for i in range(0, len(text), chunk_chars):
        yield text[i : i + chunk_chars]

for chunk in fake_stream(full, chunk_chars=24):
    sys.stdout.write(chunk)
    sys.stdout.flush()
    time.sleep(0.01)
print()

Streaming reduces percei

ved latency by surfacing

 tokens as they're gener

ated. Server-Sent Events

 (SSE) is the wire forma

...

deltas. Partial JSON par

sing lets you surface fi

elds as soon as they fin

ish, not after the whole

 object.

## 2) Async generator
Wrapping the stream in an async generator lets you compose it with concurrent work (e.g. start retrieving the next document while the current answer is still streaming).

In [4]:
import asyncio

async def astream(text: str, chunk_chars: int = 24):
    for chunk in fake_stream(text, chunk_chars):
        await asyncio.sleep(0.01)
        yield chunk

async def consume() -> str:
    out = []
    async for piece in astream(full):
        out.append(piece)
    return ''.join(out)

received = await consume()
print('received', len(received), 'chars; equal to source:', received == full)

received 320 chars; equal to source: True


## 3) Partial JSON parsing
When the model emits JSON, you usually want to render fields the moment they complete (`title`, then `authors[0]`, etc.) rather than waiting for the closing brace. We use a tiny tolerant parser that retries `json.loads` on a progressively closed copy of the buffer.

In [5]:
json_full = complete(
    model=MODEL, namespace=NS,
    messages=[
        Message(role='system', content='Extract paper metadata as JSON.'),
        Message(role='user', content='Routing-Aware Sparse Mixture-of-Experts (Garcia et al, 2024).'),
    ],
    response_format={'type': 'json_object'},
).content
print(json_full)

{"title": "Routing-Aware Sparse Mixture-of-Experts for Latency-Bounded Inference", "authors": ["A. Garcia", "K. Sato", "M. Owens"], "year": 2024}


In [6]:
import json

def try_close(buf: str):
    # Heuristic: close any open string, then any open arrays/objects, then try to parse.
    s = buf
    open_braces = s.count('{') - s.count('}')
    open_brackets = s.count('[') - s.count(']')
    # Close a dangling string by counting unescaped quotes.
    quotes = 0
    esc = False
    for c in s:
        if esc:
            esc = False
            continue
        if c == '\\':
            esc = True
        elif c == '"':
            quotes += 1
    if quotes % 2 == 1:
        s += '"'
    s += ']' * open_brackets + '}' * open_braces
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        return None

buf = ''
last_keys: set[str] = set()
for chunk in fake_stream(json_full, chunk_chars=12):
    buf += chunk
    parsed = try_close(buf)
    if isinstance(parsed, dict) and set(parsed) - last_keys:
        newly = set(parsed) - last_keys
        for k in newly:
            print(f'-> {k}: {parsed[k]!r}')
        last_keys = set(parsed)

-> title: 'R'
-> authors: ['']
-> year: 2024


## When NOT to stream
* Short responses (< 200 chars) — the UX win is invisible.
* When you need the *full* result before doing anything (e.g. JSON you have to validate with strict Pydantic — though partial parsing above gives you progress).
* When downstream consumers can't handle deltas (most logging pipelines).

## Production notes
* On the wire it's SSE: `Content-Type: text/event-stream`, `Cache-Control: no-cache`, `Connection: keep-alive`, lines beginning with `data: ` and ending with a blank line.
* `litellm.completion(..., stream=True)` returns an iterator of `ModelResponse` chunks; their async sibling is `litellm.acompletion(..., stream=True)`.
* For FastAPI streaming endpoints, see `07-deployment-patterns/00-fastapi-streaming-agent/`.